## Gold Layer — Training Summary Weekly

Weekly aggregations of training load, volume, and trends for long-term coaching insights.

**Purpose:**
* Training load analysis and recovery recommendations
* Performance trend identification over time
* Periodization and volume tracking
* Heart rate zone distribution analysis

**Source:** `garmin_lakehouse.silver.activities`

**Refresh Cadence:** Weekly (or at the end of each training week)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS garmin_lakehouse.gold;

In [0]:
%sql
CREATE OR REPLACE TABLE garmin_lakehouse.gold.training_summary_weekly AS

WITH weekly_base AS (
    SELECT
        DATE_TRUNC('WEEK', start_date) as week_start_date,
        activity_type_key,
        COUNT(*) as activity_count,
        SUM(distance_km) as total_distance_km,
        SUM(duration_minutes) as total_duration_minutes,
        SUM(elevation_gain_m) as total_elevation_gain_m,
        AVG(avg_pace_min_per_km) as avg_pace_min_per_km,
        AVG(avg_heart_rate) as avg_heart_rate,
        MAX(max_heart_rate) as max_heart_rate,
        
        -- Heart rate zones (convert seconds to minutes)
        SUM(hr_zone_1_seconds) / 60 as total_hr_zone_1_minutes,
        SUM(hr_zone_2_seconds) / 60 as total_hr_zone_2_minutes,
        SUM(hr_zone_3_seconds) / 60 as total_hr_zone_3_minutes,
        SUM(hr_zone_4_seconds) / 60 as total_hr_zone_4_minutes,
        SUM(hr_zone_5_seconds) / 60 as total_hr_zone_5_minutes,
        
        -- Training effects
        AVG(aerobic_training_effect) as avg_aerobic_effect,
        AVG(anaerobic_training_effect) as avg_anaerobic_effect,
        AVG(vo2_max) as avg_vo2_max,
        
        -- Calories
        SUM(total_calories) as total_calories,
        
        -- Running metrics
        AVG(avg_cadence_spm) as avg_cadence_spm,
        AVG(avg_stride_length_m) as avg_stride_length_m,
        SUM(total_steps) as total_steps
        
    FROM garmin_lakehouse.silver.activities
    WHERE start_date >= DATE_SUB(CURRENT_DATE(), 180)  -- Last 6 months
    GROUP BY week_start_date, activity_type_key
),

weekly_totals AS (
    SELECT
        week_start_date,
        SUM(activity_count) as total_activities,
        SUM(total_distance_km) as total_distance_km,
        SUM(total_duration_minutes) as total_duration_minutes
    FROM weekly_base
    GROUP BY week_start_date
)

SELECT
    b.week_start_date,
    DATE_ADD(b.week_start_date, 6) as week_end_date,
    b.activity_type_key,
    b.activity_count,
    t.total_activities as week_total_activities,
    
    -- Distance & Duration
    ROUND(b.total_distance_km, 2) as total_distance_km,
    ROUND(t.total_distance_km, 2) as week_total_distance_km,
    ROUND(b.total_duration_minutes, 1) as total_duration_minutes,
    ROUND(t.total_duration_minutes, 1) as week_total_duration_minutes,
    
    -- Pace & Intensity
    ROUND(b.avg_pace_min_per_km, 2) as avg_pace_min_per_km,
    ROUND(b.avg_heart_rate, 0) as avg_heart_rate,
    ROUND(b.max_heart_rate, 0) as max_heart_rate,
    
    -- HR Zone Distribution (minutes)
    ROUND(b.total_hr_zone_1_minutes, 1) as hr_zone_1_minutes,
    ROUND(b.total_hr_zone_2_minutes, 1) as hr_zone_2_minutes,
    ROUND(b.total_hr_zone_3_minutes, 1) as hr_zone_3_minutes,
    ROUND(b.total_hr_zone_4_minutes, 1) as hr_zone_4_minutes,
    ROUND(b.total_hr_zone_5_minutes, 1) as hr_zone_5_minutes,
    
    -- Training Load
    ROUND(b.avg_aerobic_effect, 1) as avg_aerobic_effect,
    ROUND(b.avg_anaerobic_effect, 1) as avg_anaerobic_effect,
    ROUND(b.avg_vo2_max, 1) as avg_vo2_max,
    
    -- Calories & Steps
    ROUND(b.total_calories, 0) as total_calories,
    ROUND(b.total_steps, 0) as total_steps,
    
    -- Running Metrics
    ROUND(b.avg_cadence_spm, 0) as avg_cadence_spm,
    ROUND(b.avg_stride_length_m, 2) as avg_stride_length_m,
    
    -- Elevation
    ROUND(b.total_elevation_gain_m, 0) as total_elevation_gain_m,
    
    -- Metadata
    CURRENT_TIMESTAMP() as loaded_at
    
FROM weekly_base b
INNER JOIN weekly_totals t ON b.week_start_date = t.week_start_date
ORDER BY b.week_start_date DESC, b.activity_type_key

In [0]:
%sql
SELECT 
    week_start_date,
    week_end_date,
    activity_type_key,
    activity_count,
    total_distance_km,
    total_duration_minutes,
    avg_pace_min_per_km,
    avg_heart_rate,
    avg_aerobic_effect,
    avg_anaerobic_effect
FROM garmin_lakehouse.gold.training_summary_weekly
ORDER BY week_start_date DESC
LIMIT 20